<font color='red'><b>**WARNING**</b></font> <br/>
어떠한 사유로도 임의로 복사, 촬영, 녹음, 복제, 보관, 전송하거나 허가 받지 않은 저장매체를 이용한 보관, 제3자에게 누설, 공개 또는 사용하는 등의 무단 사용 및 불법 배포 시 법적 조치를 받을 수 있습니다. <br/>

<div style="text-align: right; color: #7f8c8d; font-size: 0.9em; margin-top: 20px;">
📝 Author: 박사홍 (Sahong Pak)</br>
📧 Contact: sahong.pak@gmail.com</br>
📌 Version: v1.0</br>
📅 Last Updated: 2025-10-14</br>
</div>

</br>

<font color='#02e9fa'><b>**🚩 학습목표**</b></font></br>
이번 장에서는 역전파(Backpropagation) 과정에 대해 학습합니다.

역전파 과정을 직접 확인해보고, 코드로 구현하는 방법을 익혀봅시다.

</br>

## 텐서 (Tensor)
> 텐서(Tensor)는 PyTorch에서 사용하는 기본 자료구조입니다.

In [ ]:
import torch
x = torch.rand(3, 3)
y = torch.ones(3, 3)

print(x)
print(y)

</br>

## 역전파 (Backpropagation)
> 가중치 $w$와 편향 $b$를 계산하기위한 방법으로, 손실 함수의 기울기를 효율적으로 계산합니다.

$$
y = wx + b
$$

**초기 값**</br>
$x = 2$</br>
$y = 4$</br>
$w = 3$</br>
$b = 1$

In [ ]:
# TODO 1: PyTorch를 이용하여 위 초기값을 설정해봅시다.

x = torch.tensor([2.0])
y = torch.tensor([4.0])

weight = torch.tensor([3.0], requires_grad=True) # 찾아야하는 대상 (편미분 대상)
b = torch.tensor([1.0], requires_grad=True) # 찾아야하는 대상 (편미분 대상)

print(weight, b)

<font color='#02e9fa'><b>**💡 AutoGrad**</b></font></br>
PyTorch를 사용하는 이유 중 하나로, AutoGrad가 있습니다.

PyTorch에서 `.backward()` 메서드를 호출하면 그래프를 따라 자동으로 역전파를 수행하여 기울기를 구할 수 있습니다.

<font color='#02e9fa'><b>**💡 requires_grad**</b></font></br>
requires_grad 속성은 이 변수를 미분할 수 있게 해줘 라고 알려주는 속성입니다.

모델의 가중치, 편향 또는 업데이트가 필요한 변수들에 `True`로 설정하여 사용할 수 있습니다.

</br>

### 순전파 (Forward Pass)
> 순전파(Forward Pass)는 초기 값을 바탕으로 모델의 예측값을 계산하는 방법입니다.
>
> 즉, 단순히 값을 대입하여 계산하는 것을 말합니다.

$$
\hat{y} = wx + b = 3 \times 2 + 1 = 7
$$

In [ ]:
# TODO 2: 순전파를 계산해봅시다.
y_prediction = weight * x + b
print(y_prediction.item())

</br>

### 손실 계산 (Loss)
> MSE(Mean Squared Error)로 실제 값($y$)과 예측값($\hat{y}$)의 손실을 계산합니다.

$$
MSE = \frac{1}{N} \sum_{i=1}^{N}(y - \hat{y})^{2}
$$

$$
L = \frac{1}{1} \sum_{i=1}^{1}(4-7)^2 = (-3)^2 = 9
$$

In [ ]:
# TODO 3: 손실을 계산해봅시다.
loss = (y - y_prediction) ** 2
print(loss.item())

</br>

### 역전파 (Backpropagation)
> 역전파(Backpropagation)은 연쇄 법칙(Chain Rule)을 이용하여 각 파라미터에 대한 기울기를 계산합니다.
>
> 업데이트해야할 파라미터는 $w$와 $b$이므로 각각의 편미분을 계산하게됩니다.

**$\hat{y}$에 대한 편미분**

$$
\frac{\partial L}{\partial w} = \frac{\partial L}{\partial \hat{y}} \times \frac{\partial \hat{y}}{\partial w}
$$

**$b$에 대한 편미분**

$$
\frac{\partial L}{\partial b} = \frac{\partial L}{\partial \hat{y}} \times \frac{\partial \hat{y}}{\partial b}
$$

</br>

**Step 1: $\frac{\partial L}{\partial \hat{y}}$ 계산**

손실 함수(MSE)를 $\hat{y}$에 대해 미분합니다.

$y-\hat{y}$를 $u$로 치환하면 $L = u^2$ 이므로,

$$
\frac{\partial L}{\partial \hat{y}} = \frac{\partial L}{\partial u} \times \frac{\partial u}{\partial \hat{y}}
$$

$$
\frac{\partial L}{\partial u} = \frac{\partial(u^2)}{\partial u} = 2u
$$

$$
\frac{\partial u}{\partial \hat{y}} = \frac{\partial(y - \hat{y})}{\partial \hat{y}} = -1
$$

$$
\frac{\partial L}{\partial \hat{y}} = 2u \times (-1) = -2(y - \hat{y}) = -2(4 - 7) = 6
$$

</br>

**Step 2: $\frac{\partial \hat{y}}{\partial w}$ 계산**

$\hat{y} = wx + b$를 $w$에 대해 미분합니다.

$$
\frac{\partial \hat{y}}{\partial w} = x = 2
$$

</br>

**Step 3: $\frac{\partial \hat{y}}{\partial b}$ 계산**

$\hat{y} = wx + b$를 $b$에 대해 미분합니다.

$$
\frac{\partial \hat{y}}{\partial b} = 1
$$

</br>

**Step 4: 최종 기울기 계산**

$$
\frac{\partial L}{\partial w} = \frac{\partial L}{\partial \hat{y}} \times \frac{\partial \hat{y}}{\partial w} = 6 \times 2 = 12
$$

$$
\frac{\partial L}{\partial b} = \frac{\partial L}{\partial \hat{y}} \times \frac{\partial \hat{y}}{\partial b} = 6 \times 1 = 6
$$

In [ ]:
# TODO 4: 역전파를 수행해봅시다.
loss.backward()

print(f"∂L/∂w: {weight.grad.item()}")
print(f"∂L/∂b: {b.grad.item()}")

</br>

#### 파라미터 업데이트
> 학습률(Learning Rate) $\alpha = 0.1$을 사용하여 파라미터를 업데이트합니다.

$$
w_\text{new} = w - \alpha \frac{\partial L}{\partial w} = 3 - 0.1 \times 12 = 1.8
$$

$$
b_\text{new} = b - \alpha \frac{\partial L}{\partial b} = 1 - 0.1 \times 6 = 0.4
$$

In [ ]:
# TODO 5: Learning Rate를 0.1로 설정하여 파라미터를 업데이트 해봅시다.

learning_rate = 0.1

with torch.no_grad(): # 업데이트 시에는 추적 비활성화
    weight -= learning_rate * weight.grad
    b -= learning_rate * b.grad

    weight.grad.zero_()
    b.grad.zero_()

print(weight.item())
print(b.item())

<font color='#02e9fa'><b>**💡 no_grad()**</b></font></br>
`no_grad()` 메서드는 기울기를 추적하지 않도록합니다.

현재 상태에서의 기울기를 구한 값을 활용해야하기 때문에 이 메서드를 사용하지 않으면</br>
다음 기울기에 누저하여 잘못된 값을 도출하게됩니다.

</br>

#### 검증
> 새롭게 업데이트된 파라미터를 기반으로 다시 계산을 진행해봅시다.

$$
\hat{y} = wx + b = 1.8 \times 2 + 0.4 = 4
$$

$$
L = \frac{1}{1} \sum_{i=1}^{1}(4-4)^2 = (0)^2 = 0
$$

In [ ]:
# TODO 6: 새롭게 업데이트 된 파라미터를 기반으로 계산해봅시다.
y_prediction_new = weight * x + b
loss_new = (y - y_prediction_new) ** 2

print(y_prediction_new.item())
print(loss_new.item())